# ⚡ GÖP Fiyat Tahmini: Final Model, Öznitelik ve Metrik Analizi

Bu notebook, **Proje Sunumu** için özel olarak hazırlanmıştır. İçeriği:
1. Mevcut LightGBM modelimizin kullandığı **Özniteliklerin (Features) Analizi**.
2. **Matematiksel Paradoks:** Günlük Ortalama WAPE ile Global Hacim Ağırlıklı WAPE arasındaki kritik fark.
3. **Derin Öğrenme (EPNet) vs LightGBM** Karşılaştırması (Gerçek Log Dosyalarından Okunarak Hesaplanmıştır).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (14, 7)

# Import project modules
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from db.connection import get_db_engine
from sqlalchemy import text
from src.features.feature_engineering import build_robust_features, get_feature_columns


## 1. Veri Setinin Yüklenmesi ve Öznitelik Mühendisliği


In [ ]:
print("📦 Veritabanından veriler çekiliyor...")
engine = get_db_engine()
master_sql = text('''
    SELECT 
        m.ts,
        m.price_usd AS mcp_price_usd,
        m.price_try AS mcp_price_try,
        s.system_marginal_price_try AS smp_price_try,
        l.load_forecast_mw,
        k.total_mw AS kgup_total_mw,
        k.natural_gas_mw AS kgup_gas_mw,
        k.wind_mw AS kgup_wind_mw,
        k.solar_mw AS kgup_solar_mw,
        k.dammed_hydro_mw + k.river_hydro_mw AS kgup_hydro_mw,
        g.total_mw AS actual_gen_total_mw,
        c.consumption_mw AS actual_cons_mw,
        w.turkey_weighted_temperature_c AS temperature_c,
        wf.turkey_weighted_temperature_forecast_c AS temperature_forecast_c,
        mc.usd_try,
        mc.brent_oil_usd,
        ng.gas_reference_price_try AS natural_gas_grf_try
    FROM raw_mcp_hourly m
    LEFT JOIN raw_smp_hourly s ON m.ts = s.ts
    LEFT JOIN raw_load_forecast_hourly l ON m.ts = l.ts
    LEFT JOIN raw_kgup_hourly k ON m.ts = k.ts
    LEFT JOIN raw_actual_generation_hourly g ON m.ts = g.ts
    LEFT JOIN raw_actual_consumption_hourly c ON m.ts = c.ts
    LEFT JOIN raw_weather_hourly w ON m.ts = w.ts
    LEFT JOIN raw_weather_forecast_hourly wf ON m.ts = wf.ts
    LEFT JOIN raw_macro_daily mc ON DATE(m.ts) = mc.entry_date
    LEFT JOIN raw_natural_gas_daily ng ON DATE(m.ts) = ng.entry_date
    WHERE m.ts >= '2025-01-01'
    ORDER BY m.ts ASC;
''')

with engine.connect() as conn:
    df_raw = pd.read_sql(master_sql, conn)

df_raw['ts'] = pd.to_datetime(df_raw['ts']).dt.tz_convert('Europe/Istanbul')
df_raw = df_raw.set_index('ts').sort_index()
df_raw['usd_try'] = df_raw['usd_try'].ffill().bfill()
df_raw['brent_oil_usd'] = df_raw['brent_oil_usd'].ffill().bfill()
df_raw['natural_gas_grf_try'] = df_raw['natural_gas_grf_try'].ffill().bfill()

df_feat = build_robust_features(df_raw)
feature_cols = get_feature_columns('robust', df_feat)
target_col = 'mcp_price_usd'

df_model = df_feat.dropna(subset=feature_cols + [target_col]).copy()
print(f"✅ Model Veri Seti Hazır: {len(df_model)} satır, {len(feature_cols)} öznitelik.")


## 2. Mevcut Modelin Öznitelik Analizi (Feature Importance)


In [ ]:
X = df_model[feature_cols]
y = df_model[target_col].values
y_log = np.log1p(np.maximum(0, y))

params = {
    'n_estimators': 300,
    'learning_rate': 0.03,
    'max_depth': 8,
    'num_leaves': 63,
    'random_state': 42,
    'deterministic': True,
    'force_col_wise': True
}

lgb_model = lgb.LGBMRegressor(**params)
lgb_model.fit(X, y_log)

lgb.plot_importance(lgb_model, max_num_features=15, height=0.6, importance_type='gain', title="LightGBM En Önemli 15 Öznitelik")
plt.tight_layout()
plt.show()


## 3. Dinamik 1 Yıllık Global WAPE Analizi (Gerçek Log Dosyalarından)
Kendi ürettiğimiz Deep Learning (EPNet) Modellerinin sonuçlarını (MAE metriklerini) `logs` klasöründen **canlı** olarak okuyup, Veritabanından o dönemin toplam hacmini çekerek **Gerçek Global WAPE** dönüşümünü gerçekleştiriyoruz.


In [ ]:
# 1. 2025-07-29 ile 2026-07-30 arasındaki 365 günlük test periyodunun Toplam Hacmini veritabanından çekelim
print("🔄 Veritabanından test periyoduna ait Toplam Hacim çekiliyor...")
with engine.connect() as conn:
    res = conn.execute(text("SELECT SUM(price_usd) FROM raw_mcp_hourly WHERE ts >= '2025-07-29' AND ts <= '2026-07-30';")).scalar()
    total_volume_usd = float(res)

print(f"💰 1 Yıllık Test Dönemi Toplam Pazar Hacmi (Denominator): ${total_volume_usd:,.2f}")

# 2. Log dosyalarından modelleri okuyup Global WAPE'i canlı hesaplayalım
benchmark_records = []

# Canlı Sistem LightGBM (Geçmiş analizden)
benchmark_records.append({
    'Model': 'Baseline LightGBM (Log1p) [Canlı]',
    'MAE ($/MWh)': 8.436,
    'Global WAPE (%)': 16.53
})

# EPNet Robust Logs
robust_log_path = '../logs/epnet_fast_robust_experiments/epnet_robust_summary.csv'
if os.path.exists(robust_log_path):
    df_robust = pd.read_csv(robust_log_path)
    for _, row in df_robust.iterrows():
        mae = float(row.get('mae_12m', row.get('mae', 0)))
        total_abs_diff = mae * 8760
        global_wape = (total_abs_diff / total_volume_usd) * 100
        
        benchmark_records.append({
            'Model': f"EPNet - {row['name']}",
            'MAE ($/MWh)': round(mae, 3),
            'Global WAPE (%)': round(global_wape, 2)
        })

# EPNet Grid Logs (Seçili olanlar)
grid_log_path = '../logs/epnet_fast_grid_experiments_yusuf/epnet_fast_grid_summary.csv'
if os.path.exists(grid_log_path):
    df_grid = pd.read_csv(grid_log_path)
    # Sadece en iyilerden 3 tanesini alalım
    top_grids = df_grid.sort_values('mae_12m').head(3)
    for _, row in top_grids.iterrows():
        mae = float(row.get('mae_12m', row.get('mae', 0)))
        total_abs_diff = mae * 8760
        global_wape = (total_abs_diff / total_volume_usd) * 100
        
        benchmark_records.append({
            'Model': f"EPNet - {row['name']}",
            'MAE ($/MWh)': round(mae, 3),
            'Global WAPE (%)': round(global_wape, 2)
        })

df_bench = pd.DataFrame(benchmark_records).sort_values('Global WAPE (%)')
display(df_bench)

# 3. Görselleştirme
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=df_bench, x='Global WAPE (%)', y='Model', palette='magma', ax=ax)
ax.set_title('Gerçek Loglardan Hesaplanmış 1 Yıllık Global WAPE Performansı', fontsize=14)
ax.set_xlim(10, 25)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', padding=5, color='white')
plt.tight_layout()
plt.show()
